# Smart MCQ Solver

## Library Imports

In [1]:
import os
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# List files in input directory for environment verification
input_dir = '/kaggle/input'
if os.path.exists(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            print(os.path.join(root, file))


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Dummy Submission

In [2]:
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample_sub.to_csv('submission.csv', index=False)


# Milestones

## Milestone 1

### Q1

In [3]:
# Load training dataset
mcq_train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

# Find the label column name
label_column_name = 'answer' if 'answer' in mcq_train_data.columns else 'label'
label_distribution = mcq_train_data[label_column_name].value_counts()

# Print Q1 answer
print(f"Q1 Answer: {label_distribution.max() + label_distribution.min()}")


Q1 Answer: 814


### Q2

In [4]:
def normalize_text_data(raw_text):
    clean_str = str(raw_text).lower()
    clean_str = clean_str.translate(str.maketrans('', '', string.punctuation))
    return clean_str

processed_prompts = mcq_train_data['prompt'].apply(normalize_text_data)
unique_vocabulary = set()
for prompt_text in processed_prompts:
    unique_vocabulary.update(prompt_text.split())

print(f"Q2 Answer: {len(unique_vocabulary)}")


Q2 Answer: 859


### Q3

In [5]:
# Filter stop words from the first prompt
first_prompt_tokens = processed_prompts.iloc[0].split()
filtered_tokens_no_stops = [token for token in first_prompt_tokens if token not in ENGLISH_STOP_WORDS]

print(f"Q3 Answer: {len(filtered_tokens_no_stops)}")


Q3 Answer: 13


### Q4

In [6]:
# Build text clusters combining prompt and all options
mcq_options = ['A', 'B', 'C', 'D', 'E']
text_corpora = []
for idx, data_row in mcq_train_data.iterrows():
    fused_text = str(data_row['prompt']) + " " + " ".join([str(data_row[opt]) for opt in mcq_options])
    text_corpora.append(fused_text)

tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_vec.fit(text_corpora)

print(f"Q4 Answer: {len(tfidf_vec.get_feature_names_out())}")


Q4 Answer: 2762


### Q5

In [7]:
# Cosine similarity between prompt and Option A for the first row
prompt_vector = tfidf_vec.transform([str(mcq_train_data.iloc[0]['prompt'])])
option_a_vector = tfidf_vec.transform([str(mcq_train_data.iloc[0]['A'])])
similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

print(f"Q5 Answer: {similarity_score:.4f}")

# Metric calculation function for MAP@3
def calculate_map3(predictions_list, target_labels):
    ranking_scores = []
    for predictions, target in zip(predictions_list, target_labels):
        score = 0.0
        for rank_idx, pred in enumerate(predictions[:3]):
            if pred == target:
                score = 1.0 / (rank_idx + 1)
                break
        ranking_scores.append(score)
    return np.mean(ranking_scores)


Q5 Answer: 0.2720


### Q6

In [8]:
# Run similarity-based predictions pipeline on training set
tfidf_predictions = []
exact_first_choice_hits = 0

for _, data_row in mcq_train_data.iterrows():
    prompt_tfidf_vec = tfidf_vec.transform([str(data_row['prompt'])])
    similarity_map = {}
    for opt in mcq_options:
        opt_tfidf_vec = tfidf_vec.transform([str(data_row[opt])])
        similarity_map[opt] = cosine_similarity(prompt_tfidf_vec, opt_tfidf_vec)[0][0]
    
    # Sort option labels descending by similarity score
    sorted_options = sorted(similarity_map, key=similarity_map.get, reverse=True)
    tfidf_predictions.append(sorted_options)
    
    if sorted_options[0] == data_row[label_column_name]:
        exact_first_choice_hits += 1

accuracy_percentage = (exact_first_choice_hits / len(mcq_train_data)) * 100
print(f"Q6 Answer: {accuracy_percentage:.2f}%")


Q6 Answer: 13.55%


### Q7

In [9]:
def compute_single_map3(prediction_list, target_label):
    for rank_idx, pred in enumerate(prediction_list[:3]):
        if pred == target_label:
            return 1.0 / (rank_idx + 1)
    return 0.0

print(f"Question 7 Verification Output: {compute_single_map3(['C', 'A', 'B'], 'C')}")


Question 7 Verification Output: 1.0


### Q8

In [10]:
print(f"Question 8 Verification Output: {compute_single_map3(['D', 'B', 'E'], 'B')}")


Question 8 Verification Output: 0.5


### Q9

In [11]:
# Evaluate majority class baseline MAP@3
top_three_labels = list(label_distribution.index[:3])
majority_baseline_preds = [top_three_labels] * len(mcq_train_data)
majority_class_map3 = calculate_map3(majority_baseline_preds, mcq_train_data[label_column_name])

print(f"Q9 Answer: {majority_class_map3:.4f}")


Q9 Answer: 0.4213


### Q10

In [12]:
# Evaluate TF-IDF predictions MAP@3
tfidf_pipeline_map3 = calculate_map3(tfidf_predictions, mcq_train_data[label_column_name])

print(f"Q10 Answer: {tfidf_pipeline_map3:.4f}")


Q10 Answer: 0.2962
